# Outlier Detection and Data Quality

This notebook cleans the sold listings dataset by checking key numeric fields, handling invalid negative values, capping extreme values, and creating a filtered dataset for analysis.

The goal is to reduce the impact of unrealistic or highly extreme values without permanently changing the original raw dataset.

## 1. Import libraries and load the data

The original notebook used a local file path from one computer. This version stores the file paths in variables so they are easier to update and reuse.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

# Update these paths if your files are stored somewhere else.
INPUT_PATH = "sold_feature_engineered.csv"
OUTPUT_PATH = "sold_no_outliers.csv"

sold = pd.read_csv(INPUT_PATH)

sold.head()

/var/folders/13/041lbv8n5d1clk4p9mt3qr540000gp/T/ipykernel_63386/3542488925.py:10: DtypeWarning: Columns (2,7,45,60,78,79,80,81,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv(INPUT_PATH)


,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,price_per_sqft,days_on_market,year,month,YrMo,listing_to_contract_days,contract_to_close_days,extreme_price_ratio_flag,negative_contract_to_close_flag,negative_listing_to_contract_flag
0,"Carpet,Wood",True,NaN,NaN,True,98000.0,556366533,michellefsellsoc@gmail.com,2022-02-25,95000.0,...,69.444444,86.0,2022,2,2022-02,107.0,28.0,False,False,False
1,NaN,False,NaN,NaN,False,1200.0,556366530,dineshcalre@gmail.com,2022-02-19,1200.0,...,1.411765,125.0,2022,2,2022-02,129.0,0.0,False,False,False
2,NaN,True,NaN,NaN,False,1100000.0,556366044,cindydavishomes@gmail.com,2022-04-15,1100000.0,...,818.452381,106.0,2022,4,2022-04,113.0,71.0,False,False,False
3,NaN,True,NaN,NaN,False,2499999.0,556365765,bryanmeathe@gmail.com,2022-01-04,2499999.0,...,945.179206,37.0,2022,1,2022-01,37.0,46.0,False,False,False
4,"Carpet,Tile",NaN,NaN,NaN,NaN,598888.0,556365290,steven@westsideres.com,2022-01-12,640000.0,...,534.223706,15.0,2022,1,2022-01,65.0,26.0,False,False,False


## 2. Define the numeric fields to clean

These are the columns where invalid values or extreme outliers could distort the analysis.

In [4]:
KEY_NUMERIC_FIELDS = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "LotSizeSquareFeet",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "DaysOnMarket",
    "YearBuilt",
]

# Keep only fields that actually exist in the dataset.
missing_fields = [field for field in KEY_NUMERIC_FIELDS if field not in sold.columns]
available_fields = [field for field in KEY_NUMERIC_FIELDS if field in sold.columns]

print(f"Rows: {sold.shape[0]:,}")
print(f"Columns: {sold.shape[1]:,}")
print(f"Available numeric fields: {available_fields}")

if missing_fields:
    print(f"Missing fields: {missing_fields}")

Rows: 694,719
Columns: 110
Available numeric fields: ['ClosePrice', 'ListPrice', 'OriginalListPrice', 'LivingArea', 'LotSizeSquareFeet', 'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']


## 3. Review summary statistics before cleaning

This gives a baseline before any filtering or capping is applied.

In [5]:
sold[available_fields].describe().T

,count,mean,std,min,25%,50%,75%,max
ClosePrice,694719.0,863421.433725,4.535566e+06,0.0,95000.0,639000.0,1062500.0,9.895000e+08
ListPrice,694719.0,835291.800625,1.232765e+06,0.0,99900.0,638500.0,1049888.0,1.375000e+08
OriginalListPrice,693712.0,901116.414811,5.859832e+06,0.0,99000.0,648000.0,1059000.0,1.390000e+09
LivingArea,694719.0,5763.421987,1.889249e+06,0.0,1182.0,1577.0,2146.0,9.090991e+08
LotSizeSquareFeet,627229.0,497870.456666,5.177385e+07,0.0,5064.0,7194.0,11761.0,3.667752e+10
BedroomsTotal,693328.0,3.082427,1.365131e+00,0.0,2.0,3.0,4.0,1.230000e+02
BathroomsTotalInteger,694608.0,2.491106,1.348001e+00,0.0,2.0,2.0,3.0,1.750000e+02
DaysOnMarket,694719.0,36.679660,5.305953e+01,-288.0,8.0,19.0,47.0,1.243000e+04
YearBuilt,691887.0,1978.592381,2.667239e+01,1776.0,1960.0,1979.0,1999.0,2.026000e+03


In [6]:
for field in available_fields:
    median_value = sold[field].median()
    print(f"Median of {field}: {median_value:,.2f}")

Median of ClosePrice: 639,000.00
Median of ListPrice: 638,500.00
Median of OriginalListPrice: 648,000.00
Median of LivingArea: 1,577.00
Median of LotSizeSquareFeet: 7,194.00
Median of BedroomsTotal: 3.00
Median of BathroomsTotalInteger: 2.00
Median of DaysOnMarket: 19.00
Median of YearBuilt: 1,979.00


## 4. Remove invalid negative values

For these housing-related fields, negative values usually indicate data quality problems. This step removes rows where any selected numeric field is negative.

The original version accidentally reset the filtered data inside the loop, so only the final field was actually being checked. This version builds one combined filter across all numeric fields.

In [7]:
nonnegative_mask = (sold[available_fields] >= 0).all(axis=1)
sold_nonnegative = sold.loc[nonnegative_mask].copy()

rows_removed = len(sold) - len(sold_nonnegative)
print(f"Rows removed because of negative values: {rows_removed:,}")
print(f"Rows remaining: {len(sold_nonnegative):,}")

Rows removed because of negative values: 72,637
Rows remaining: 622,082


## 5. Cap extreme values using percentiles

Before using the IQR method, this step caps extremely high values based on field-specific percentiles. This keeps unusually large values from overly stretching the IQR range.

Capping does not delete rows. It only limits values above the selected percentile.

In [8]:
PERCENTILE_CAPS = {
    "ClosePrice": 0.99,
    "ListPrice": 0.99,
    "OriginalListPrice": 0.99,
    "LivingArea": 0.99,
    "LotSizeSquareFeet": 0.95,
    "BedroomsTotal": 0.999,
    "BathroomsTotalInteger": 0.999,
    "DaysOnMarket": 0.99,
    "YearBuilt": 0.999,
}

sold_capped = sold_nonnegative.copy()
cap_summary = []

for field, percentile in PERCENTILE_CAPS.items():
    if field not in available_fields:
        continue

    upper_bound = sold_capped[field].quantile(percentile)
    values_above_cap = (sold_capped[field] > upper_bound).sum()

    sold_capped[field] = sold_capped[field].clip(upper=upper_bound)

    cap_summary.append({
        "field": field,
        "percentile": percentile,
        "upper_bound": upper_bound,
        "values_capped": values_above_cap,
    })

pd.DataFrame(cap_summary)

,field,percentile,upper_bound,values_capped
0,ClosePrice,0.990,4999000.0,6219
1,ListPrice,0.990,4999000.0,6185
2,OriginalListPrice,0.990,5300000.0,6191
3,LivingArea,0.990,5551.0,6219
4,LotSizeSquareFeet,0.950,121388.2,31105
5,BedroomsTotal,0.999,13.0,605
6,BathroomsTotalInteger,0.999,13.0,573
7,DaysOnMarket,0.990,225.0,6168
8,YearBuilt,0.999,2025.0,331


## 6. Filter outliers using the IQR method

The Interquartile Range method flags values outside this range:

\[
\text{Lower Bound} = Q1 - 1.5 \times IQR
\]

\[
\text{Upper Bound} = Q3 + 1.5 \times IQR
\]

Instead of turning individual outlier values into missing values, this version removes rows that are outliers in any of the selected fields. This creates a clean analysis dataset.

In [9]:
def get_iqr_bounds(series: pd.Series, multiplier: float = 1.5) -> tuple[float, float]:
    """Return the lower and upper IQR bounds for a numeric pandas Series."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - multiplier * iqr
    upper_bound = q3 + multiplier * iqr
    return lower_bound, upper_bound


iqr_mask = pd.Series(True, index=sold_capped.index)
iqr_summary = []

for field in available_fields:
    lower_bound, upper_bound = get_iqr_bounds(sold_capped[field])
    field_mask = sold_capped[field].between(lower_bound, upper_bound, inclusive="both")
    outlier_count = (~field_mask).sum()

    iqr_mask &= field_mask

    iqr_summary.append({
        "field": field,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outliers_flagged": outlier_count,
    })

sold_filtered = sold_capped.loc[iqr_mask].copy()

pd.DataFrame(iqr_summary)

,field,lower_bound,upper_bound,outliers_flagged
0,ClosePrice,-1225000.0,2495000.0,32572
1,ListPrice,-1205000.0,2475000.0,32887
2,OriginalListPrice,-1211000.0,2485000.0,34417
3,LivingArea,-293.5,3702.5,27939
4,LotSizeSquareFeet,-4986.5,21809.5,95847
5,BedroomsTotal,-1.0,7.0,2417
6,BathroomsTotalInteger,0.5,4.5,30152
7,DaysOnMarket,-50.5,105.5,45125
8,YearBuilt,1900.5,2056.5,1339


## 7. Compare dataset size before and after cleaning

In [10]:
cleaning_summary = pd.DataFrame({
    "step": ["original", "after removing negative values", "after percentile capping", "after IQR filtering"],
    "rows": [len(sold), len(sold_nonnegative), len(sold_capped), len(sold_filtered)],
})

cleaning_summary["rows_removed_from_previous_step"] = cleaning_summary["rows"].shift(1) - cleaning_summary["rows"]
cleaning_summary["rows_removed_from_previous_step"] = cleaning_summary["rows_removed_from_previous_step"].fillna(0).astype(int)

cleaning_summary

,step,rows,rows_removed_from_previous_step
0,original,694719,0
1,after removing negative values,622082,72637
2,after percentile capping,622082,0
3,after IQR filtering,450659,171423


## 8. Review summary statistics after cleaning

In [11]:
sold_filtered[available_fields].describe().T

,count,mean,std,min,25%,50%,75%,max
ClosePrice,450659.0,679862.426094,564856.825931,0.0,17250.0,645000.0,990000.0,2495000.0
ListPrice,450659.0,669850.231299,551984.582813,450.0,16255.5,640000.0,979000.0,2475000.0
OriginalListPrice,450659.0,675368.466873,557737.938145,0.0,16000.0,649000.0,988800.0,2485000.0
LivingArea,450659.0,1656.886647,632.794014,0.0,1200.0,1568.0,2032.0,3702.0
LotSizeSquareFeet,450659.0,6782.393776,4078.862766,0.0,4500.0,6402.0,8250.0,21809.0
BedroomsTotal,450659.0,3.077615,0.983987,0.0,2.0,3.0,4.0,7.0
BathroomsTotalInteger,450659.0,2.305102,0.787734,1.0,2.0,2.0,3.0,4.0
DaysOnMarket,450659.0,25.427456,24.212500,0.0,7.0,16.0,37.0,105.0
YearBuilt,450659.0,1975.624619,26.940633,1901.0,1956.0,1976.0,1995.0,2025.0


In [12]:
for field in available_fields:
    median_value = sold_filtered[field].median()
    print(f"Median of {field}: {median_value:,.2f}")

Median of ClosePrice: 645,000.00
Median of ListPrice: 640,000.00
Median of OriginalListPrice: 649,000.00
Median of LivingArea: 1,568.00
Median of LotSizeSquareFeet: 6,402.00
Median of BedroomsTotal: 3.00
Median of BathroomsTotalInteger: 2.00
Median of DaysOnMarket: 16.00
Median of YearBuilt: 1,976.00


## 9. Save the cleaned dataset

In [14]:
sold_filtered.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")

Saved cleaned dataset to: sold_no_outliers.csv
